In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import sklearn

In [13]:
# Test uploading a CSV file
csv = pd.read_csv('../../data/basketball-reference/players(25-26)/regular-season/adjusted-shooting.csv', header=1)
df = pd.DataFrame(csv)
df

,Rk,Player,Age,Team,Pos,G,GS,MP,FG%,2P%,...,3P+,eFG+,FT+,TS+,FTr+,3PAr+,FG Add,TS Add,Awards,-9999
0,1.0,Amen Thompson,23.0,HOU,PG,79.0,79.0,2953.0,0.534,0.573,...,60.0,100.0,100.0,102.0,142.0,27.0,-0.3,29.5,DPOY-8,thompam01
1,2.0,Kevin Durant,37.0,HOU,SF,78.0,78.0,2840.0,0.520,0.572,...,115.0,108.0,112.0,110.0,129.0,79.0,116.4,187.0,CPOY-9ASNBA2,duranke01
2,3.0,Desmond Bane,27.0,ORL,SG,82.0,82.0,2756.0,0.484,0.535,...,109.0,101.0,116.0,104.0,109.0,85.0,18.0,68.8,NaN,banede01
3,4.0,Toumani Camara,25.0,POR,PF,82.0,82.0,2731.0,0.440,0.579,...,103.0,103.0,90.0,99.0,58.0,160.0,30.6,-6.2,NaN,camarto01
4,5.0,Jabari Smith Jr.,22.0,HOU,PF,77.0,77.0,2705.0,0.450,0.537,...,101.0,99.0,99.0,98.0,81.0,121.0,-9.9,-24.5,NaN,smithja05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
729,579.0,Colby Jones,23.0,DET,SG,1.0,0.0,7.0,0.333,1.000,...,0.0,61.0,NaN,57.0,0.0,161.0,-1.3,-1.5,NaN,jonesco02
730,580.0,Noa Essengue,19.0,CHI,PF,2.0,0.0,6.0,0.000,0.000,...,0.0,0.0,NaN,0.0,0.0,161.0,-3.3,-3.5,NaN,essenno01
731,581.0,Trentyn Flowers,20.0,CHI,SF,2.0,0.0,6.0,0.667,1.000,...,0.0,122.0,NaN,115.0,0.0,80.0,0.7,0.5,NaN,flowetr01
732,582.0,Darius Brown II,26.0,CLE,SG,1.0,0.0,3.0,0.000,0.000,...,NaN,0.0,NaN,0.0,0.0,0.0,-1.1,-1.2,NaN,brownda04


In [ ]:
from pathlib import Path
from functools import reduce

folder = Path('../../data/basketball-reference/players(25-26)/regular-season')

dfs = []
header1csv = ['adjusted-shooting.csv', 'play-by-play.csv', 'shooting.csv']
for csv_path in folder.glob('*.csv'):
    header_row = 1 if csv_path.name in header1csv else 0
    df = pd.read_csv(csv_path, header=header_row) #header = 1 since basketball reference
    df = df.rename(columns={'-9999': 'player_id', 'Player-additional': 'player_id'})
    df = df.drop_duplicates(subset='player_id', keep='first') # traded players get a row per team + an aggregate "NTM" row; keep the aggregate
    dataset_name = csv_path.stem
    df = df.rename(columns={col: f'{dataset_name}_{col}' for col in df.columns if col not in ('player_id', 'Player')})
    dfs.append((dataset_name, df))
    # Use to debug imports if you want
    # print(csv_path)
    # print(df.head)

print(dfs)

In [ ]:
# Merged_DF. Done with Claude
merged_df = reduce(
    lambda left, right: pd.merge(left, right, on=['player_id', 'Player'], how='outer'),
    [df for _, df in dfs]
)
merged_df

In [12]:
output_path = Path('../../data/processed/reg-szn/player_data_merged.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
merged_df.to_csv(output_path, index=False)

In [ ]:
# READ COMMENTS FROM PART 1 IF CONFUSED
from pathlib import Path
from functools import reduce

folder = Path('../../data/basketball-reference/players(25-26)/playoffs')

dfs = []
header1csv = ['adjusted-shooting.csv', 'play-by-play.csv', 'shooting.csv']
for csv_path in folder.glob('*.csv'):
    header_row = 1 if csv_path.name in header1csv else 0
    df = pd.read_csv(csv_path, header=header_row)
    df = df.rename(columns={'-9999': 'player_id', 'Player-additional': 'player_id'})
    df = df.drop_duplicates(subset='player_id', keep='first') # traded players get a row per team + an aggregate "NTM" row; keep the aggregate
    dataset_name = csv_path.stem
    df = df.rename(columns={col: f'{dataset_name}_{col}' for col in df.columns if col not in ('player_id', 'Player')})
    dfs.append((dataset_name, df))
    # print(csv_path)
    # print(df.head)

print(dfs)

merged_df = reduce(
    lambda left, right: pd.merge(left, right, on=['player_id', 'Player'], how='outer'),
    [df for _, df in dfs]
)
merged_df

output_path = Path('../../data/processed/playoffs/player_data_merged.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
merged_df.to_csv(output_path, index=False)